# 02 — Preprocessing & Feature Engineering

Prepares the Ames Housing dataset for modeling.  
Covers: train/val/test split, missing value handling (structural, MAR, MCAR), feature engineering, and categorical grouping.

**Scope:** Preprocessing only. No EDA plots or modeling.  
**Output:** Processed CSVs saved to `../data/processed/`.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 2. Data Loading

In [2]:
data_path = '../data/raw/AmesHousing.csv'
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')

Dataset shape: (2930, 82)


## 3. Train / Validation / Test Split

The partition is performed before any data-dependent preprocessing so that all subsequent statistics are estimated on the training subset only. This ordering is methodologically necessary to prevent information leakage from validation or test observations into feature construction and imputation rules.

The train subset is used to estimate reusable quantities (for example, medians, modes, thresholds, and grouping rules), and those fitted quantities are then applied unchanged to validation and test sets. This mirrors the deployment setting, where preprocessing parameters are learned once and applied to unseen data.

In [3]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=40
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (2051, 81) (2051,)
Validation: (439, 81) (439,)
Test: (440, 81) (440,)


These dimensions confirm that disjoint train/validation/test subsets were created using the specified proportions. The split therefore supports unbiased model selection (validation) and final generalization assessment (test) while preserving a dedicated training base for all fitted preprocessing decisions.

In [4]:
train_model = df.loc[X_train.index].copy()
val_model   = df.loc[X_val.index].copy()
test_model  = df.loc[X_test.index].copy()

## 4. Missing Value Handling — Structural

Variables where NA reflects absence of the feature (e.g., no garage, no basement).  
Categoricals → `'None'`; numerics → `0`.

This treatment is justified when missingness represents a semantically valid state (feature not present) rather than measurement failure. Encoding structural categorical missingness as `'None'` preserves the categorical meaning explicitly, while encoding structural numeric missingness as `0` preserves the quantitative interpretation of absence (for example, zero area or zero capacity).

In [13]:
structural_categorical = [
    'Alley',
    'Mas Vnr Type',
    'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
    'BsmtFin Type 1', 'BsmtFin Type 2',
    'Fireplace Qu',
    'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond',
    'Pool QC', 'Fence', 'Misc Feature'
]

structural_numeric = [
    'Mas Vnr Area',
    'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
    'Bsmt Full Bath', 'Bsmt Half Bath',
    'Garage Yr Blt', 'Garage Cars', 'Garage Area'
]

for df_ in [train_model, val_model, test_model]:
    for col in structural_categorical:
        if col in df_.columns:
            df_[col] = df_[col].fillna('None')
    for col in structural_numeric:
        if col in df_.columns:
            df_[col] = df_[col].fillna(0)

print("Structural missing values filled.")

Structural missing values filled.


This result confirms that the predefined structural fields were harmonized across splits using a semantics-preserving encoding. The decision is supported by domain interpretation of these variables as optional property components rather than incomplete records.

## 5. Missing Value Handling — Non-Structural

- **Lot Frontage**: MAR → impute by neighborhood median (calculated on train only).
- **Electrical**: 1 missing → assume MCAR, impute with train mode.

These imputations are targeted and mechanism-aware rather than generic. `Lot Frontage` is imputed conditionally on `Neighborhood` because frontage is context-dependent and neighborhood medians provide a local central tendency with lower distortion than a single global value. `Electrical` has only a very small number of missing observations, so train-mode imputation is a low-variance, minimally invasive correction.

Both strategies are fitted from train data only and transferred to validation/test unchanged, preserving leakage control.

In [6]:
# Lot Frontage: MAR → impute by neighborhood median
lot_frontage_medians = train_model.groupby('Neighborhood')['Lot Frontage'].median()
global_lot_frontage_median = train_model['Lot Frontage'].median()


def impute_lot_frontage(df, medians, global_median):
    df = df.copy()
    missing_idx = df['Lot Frontage'].isna()
    df.loc[missing_idx, 'Lot Frontage'] = (
        df.loc[missing_idx, 'Neighborhood']
        .map(medians)
        .fillna(global_median)
    )
    return df


train_model = impute_lot_frontage(train_model, lot_frontage_medians, global_lot_frontage_median)
val_model   = impute_lot_frontage(val_model,   lot_frontage_medians, global_lot_frontage_median)
test_model  = impute_lot_frontage(test_model,  lot_frontage_medians, global_lot_frontage_median)

# Electrical: MCAR → impute with train mode
electrical_mode = train_model['Electrical'].mode()[0]

for df_ in [train_model, val_model, test_model]:
    if 'Electrical' in df_.columns:
        df_['Electrical'] = df_['Electrical'].fillna(electrical_mode)

print("Non-structural missing values filled.")

Non-structural missing values filled.


This output confirms completion of non-structural imputation under two distinct missingness assumptions (MAR for `Lot Frontage`, near-MCAR sparse missingness for `Electrical`). The preprocessing choice is therefore aligned with variable-specific missingness behavior rather than a one-rule approach.

## 6. Feature Engineering

Derived features based on domain knowledge. Definitions are applied identically across all splits.

Each engineered variable is intended to encode a higher-level housing concept that is not fully captured by a single raw field:

- `Has_Garage`: binary availability signal that separates structural absence from positive garage capacity.
- `Total_SF`: aggregate indoor size proxy combining basement and above-ground living area.
- `House_Age`: property age at sale time, representing depreciation/obsolescence effects.
- `Years_Since_Remod`: recency of renovation, capturing modernization status beyond original build year.
- `Total_Bath`: effective sanitation capacity using full and half-bath weighting.
- `Qual_Cond`: interaction of overall quality and condition to represent joint structural-standard intensity.

These constructions retain original semantics while improving statistical expressiveness and interpretability for downstream models.

In [7]:
for df_ in [train_model, val_model, test_model]:
    # Garage existence indicator (derived from structural imputation)
    df_['Has_Garage'] = (df_['Garage Area'] > 0).astype(int)

    # Total living area
    df_['Total_SF'] = df_['Total Bsmt SF'] + df_['Gr Liv Area']

    # Age at time of sale
    df_['House_Age'] = df_['Yr Sold'] - df_['Year Built']

    # Years since last remodel
    df_['Years_Since_Remod'] = df_['Yr Sold'] - df_['Year Remod/Add']

    # Total bathrooms (full + half weighted)
    df_['Total_Bath'] = (
        df_['Full Bath'] + 0.5 * df_['Half Bath'] +
        df_['Bsmt Full Bath'] + 0.5 * df_['Bsmt Half Bath']
    )

    # Quality × Condition interaction
    df_['Qual_Cond'] = df_['Overall Qual'] * df_['Overall Cond']

    # Log-transformed target (right-skewed)
    df_['log_SalePrice'] = np.log(df_['SalePrice'])

print("Feature engineering complete.")
train_model[['Total_SF', 'House_Age', 'Years_Since_Remod', 'Total_Bath', 'Qual_Cond', 'log_SalePrice']].describe()

Feature engineering complete.


,Total_SF,House_Age,Years_Since_Remod,Total_Bath,Qual_Cond,log_SalePrice
count,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000
mean,2556.921989,36.232082,23.346173,2.225987,33.891760,12.026308
std,832.564179,30.413825,20.936352,0.813905,9.089151,0.405545
min,612.000000,-1.000000,-2.000000,1.000000,1.000000,9.480368
25%,2004.000000,7.000000,4.000000,2.000000,30.000000,11.774905
50%,2446.000000,34.000000,15.000000,2.000000,35.000000,11.995352
75%,2999.500000,54.000000,42.000000,2.500000,40.000000,12.271977
max,11752.000000,136.000000,60.000000,7.000000,90.000000,13.534473


The summary confirms that engineered variables were created successfully and exhibit plausible numeric ranges on the training split. This supports internal consistency of the feature definitions before encoding or modeling stages.

## 7. Categorical Grouping

Rare-level consolidation and ordinal simplification. All grouping rules are fitted on the train set only.

Grouping is performed to reduce sparse levels that can destabilize estimates and inflate variance in encoded design matrices. Consolidating rare categories improves statistical robustness, while ordinal collapsing (for kitchen quality) and semantic simplification (for building type and neighborhood tiers) improve interpretability without introducing leakage.

Rules are learned on train and then applied to validation/test to preserve comparability and methodological correctness.

In [8]:
def group_rare_from_train(train_series, other_series_list=None, threshold=0.05):
    """Group rare levels (< threshold) into 'Other'. Fitted on train only."""
    freq = train_series.value_counts(normalize=True)
    rare_levels = freq[freq < threshold].index.tolist()

    train_grouped = train_series.apply(lambda x: 'Other' if x in rare_levels else x)
    grouped_others = [
        s.apply(lambda x: 'Other' if x in rare_levels else x)
        for s in (other_series_list or [])
    ]
    return train_grouped, grouped_others, rare_levels


# Neighborhood
train_model['Neighborhood_grouped'], grouped_sets, _ = group_rare_from_train(
    train_model['Neighborhood'],
    [val_model['Neighborhood'], test_model['Neighborhood']],
    threshold=0.05
)
val_model['Neighborhood_grouped']  = grouped_sets[0]
test_model['Neighborhood_grouped'] = grouped_sets[1]


# Neighborhood simplified (high-value vs other)
def simplify_neighborhood(series):
    high_value = ['NridgHt', 'Somerst']
    return series.apply(lambda x: 'High' if x in high_value else 'Other')

for df_ in [train_model, val_model, test_model]:
    df_['Neighborhood_simple'] = simplify_neighborhood(df_['Neighborhood_grouped'])


# Kitchen Quality grouped (ordinal collapse)
def group_kitchen_qual(series):
    mapping = {'Po': 'Low', 'Fa': 'Low', 'TA': 'Medium', 'Gd': 'High', 'Ex': 'High'}
    return series.map(mapping).fillna(series)

for df_ in [train_model, val_model, test_model]:
    df_['Kitchen_Qual_grouped'] = group_kitchen_qual(df_['Kitchen Qual'])


# Building Type simplified
def simplify_bldg_type(series):
    return series.apply(lambda x: 'Detached' if x == '1Fam' else 'Other')

for df_ in [train_model, val_model, test_model]:
    df_['Bldg_Type_simple'] = simplify_bldg_type(df_['Bldg Type'])

print("Categorical grouping complete.")

Categorical grouping complete.


This confirms that grouped categorical representations were generated consistently across all splits using train-fitted rules. The resulting variables are expected to be less sparse and easier to interpret than the original high-granularity categories.

## 8. Price Tier Target Variable (Classification)

Tier thresholds are computed from the train set only.

Train-derived quantile thresholds define ordinal price strata without exposing validation/test distributional information. This preserves out-of-sample integrity while producing a classification target (`price_tier` and `is_high`) that is consistent with the continuous sale-price scale.

In [9]:
q1, q2 = train_model['SalePrice'].quantile([1/3, 2/3])


def assign_price_tier(price, q1, q2):
    if price <= q1: return 'Low'
    elif price <= q2: return 'Medium'
    return 'High'


for df_ in [train_model, val_model, test_model]:
    df_['price_tier'] = df_['SalePrice'].apply(lambda x: assign_price_tier(x, q1, q2))
    df_['is_high'] = (df_['price_tier'] == 'High').astype(int)

print("Price tier distribution (train):")
print(train_model['price_tier'].value_counts())
print("\nis_high distribution (train):")
print(train_model['is_high'].value_counts(normalize=True))

Price tier distribution (train):
price_tier
Low       705
High      676
Medium    670
Name: count, dtype: int64

is_high distribution (train):
is_high
0    0.670405
1    0.329595
Name: proportion, dtype: float64


These distributions verify that the train-based thresholding produced coherent class assignments and a well-defined binary high-price indicator. The target construction is therefore reproducible and leakage-controlled.

## 9. Shape Checks and Missing Value Verification

Final integrity checks are required to ensure preprocessing produced structurally consistent datasets for modeling. Shape validation confirms that split membership and feature generation are coherent; missing-value validation confirms that required modeling fields are complete after all imputations and transformations.

In [10]:
print("=== Dataset shapes ===")
print(f"Train:      {train_model.shape}")
print(f"Validation: {val_model.shape}")
print(f"Test:       {test_model.shape}")

=== Dataset shapes ===
Train:      (2051, 95)
Validation: (439, 95)
Test:       (440, 95)


In [11]:
# Columns used in modeling
modeling_features = [
    'Overall Qual', 'Total_SF', 'Lot Area', 'Garage Area', 'Total_Bath',
    'House_Age', 'Neighborhood_grouped', 'Neighborhood_simple',
    'Kitchen_Qual_grouped', 'Bldg_Type_simple',
    'log_SalePrice', 'SalePrice', 'is_high', 'price_tier'
]

print("=== Missing values in modeling features ===")
for split_name, df_ in [('Train', train_model), ('Val', val_model), ('Test', test_model)]:
    missing = df_[modeling_features].isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f"{split_name}: ✓ No missing values")
    else:
        print(f"{split_name}: ✗ Missing values found:\n{missing}")

=== Missing values in modeling features ===
Train: ✓ No missing values
Val: ✓ No missing values
Test: ✓ No missing values


If all listed modeling features are complete across train/validation/test, the preprocessing pipeline has met its immediate data-quality objective for downstream estimators.

## 10. Export Processed Datasets

In [ ]:
import os

os.makedirs('../data/processed', exist_ok=True)

train_model.to_csv('../data/processed/train.csv', index=False)
val_model.to_csv('../data/processed/val.csv', index=False)
test_model.to_csv('../data/processed/test.csv', index=False)

print("Processed datasets saved to ../data/processed/")
print(f"  train.csv  — {train_model.shape}")
print(f"  val.csv    — {val_model.shape}")
print(f"  test.csv   — {test_model.shape}")

## Conclusion
The preprocessing pipeline applies train-first methodology throughout: split before fitting any data-dependent rule, estimate imputation/grouping/threshold parameters on train only, and transfer those rules unchanged to validation and test subsets. Structural and non-structural missingness are handled with mechanism-consistent strategies, engineered features encode interpretable domain constructs, and categorical grouping reduces sparsity while preserving meaning.

Final shape and missingness checks, followed by dataset export, confirm that the resulting tables are methodologically consistent and ready for modeling.
